# Verified Reasoning Monitoring: development smoke

This notebook provisions the pinned Linux verifier first, then runs only the registered 32-task by four-candidate development feasibility check. It does not evaluate H1-H3. A failed setup is an operational result, not a scientific feasibility failure.

In [ ]:
from google.colab import files, userdata
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import hashlib
import json
import os
import re
import shutil
import subprocess
import tarfile

PROJECT_GIT_URL = ''  # Set to the repository HTTPS URL after an immutable commit exists.
PROJECT_GIT_REV = ''  # Exact 40-character commit; branches and tags are rejected.
PROJECT = Path('/content/verified-reasoning-monitoring')
RUNTIME = Path('/content/vrm-runtime')
TOOLS = RUNTIME / 'tools'
CACHE_BASE = RUNTIME / 'mathlib4-v3-official'
CACHE = RUNTIME / 'mathlib4-v3-disposable'
PREPARED_ARCHIVE_NAME = 'vrm-prepared-v2-approved.tar.gz'
PREPARED = Path('/content/vrm-artifacts/prepared-v2')
RUN = Path('/content/vrm-artifacts/development-smoke-v1')

assert re.fullmatch(r'^[0-9a-f]{40}$', PROJECT_GIT_REV), 'Use an exact project commit.'
assert PROJECT_GIT_URL.startswith('https://github.com/'), 'Use the reviewable HTTPS repository.'
hf_token = userdata.get("HF_TOKEN")
assert hf_token, 'Add HF_TOKEN through the Colab Secrets panel.'
os.environ['HF_TOKEN'] = hf_token

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', PROJECT_GIT_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', PROJECT_GIT_REV], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', PROJECT_GIT_REV], check=True)
head = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
assert head == PROJECT_GIT_REV
project_status = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'status', '--porcelain=v1', '--untracked-files=all'],
    text=True,
).strip()
assert not project_status, 'project checkout is not clean'
os.chdir(PROJECT)


## 1. Provision pinned verifier dependencies

This cell does not load Gemma. It installs the exact Lean toolchain, Landrun and Comparator, checks out the frozen Mathlib commit, and restores Mathlib's official Azure build cache. The unavailable LeanDojo cache is not retried. The approved amendment and every source revision are verified before use.


In [ ]:
# provision-verifier
def run(command, *, cwd=None, env=None):
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True)

def exact_checkout(url, revision, destination):
    if not destination.exists():
        run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)])
    run(['git', '-C', str(destination), 'fetch', 'origin', revision])
    run(['git', '-C', str(destination), 'checkout', '--detach', revision])
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == revision
    origin = subprocess.check_output(
        ['git', '-C', str(destination), 'config', '--get', 'remote.origin.url'],
        text=True,
    ).strip()
    assert origin.removesuffix('.git').rstrip('/') == url.removesuffix('.git').rstrip('/')
    status = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    assert not status, f'source checkout is not clean: {destination}'

RUNTIME.mkdir(parents=True, exist_ok=True)
TOOLS.mkdir(parents=True, exist_ok=True)
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'curl', 'elan', 'git', 'util-linux'])

go_version = 'go1.24.0'
go_metadata = json.load(urlopen('https://go.dev/dl/?mode=json&include=all'))
go_file = next(
    file
    for release in go_metadata if release['version'] == go_version
    for file in release['files']
    if file['os'] == 'linux' and file['arch'] == 'amd64' and file['kind'] == 'archive'
)
go_archive = RUNTIME / go_file['filename']
if not go_archive.exists():
    urlretrieve('https://go.dev/dl/' + go_file['filename'], go_archive)
assert hashlib.sha256(go_archive.read_bytes()).hexdigest() == go_file['sha256']
go_root = TOOLS / go_version
if not go_root.exists():
    go_root.mkdir()
    with tarfile.open(go_archive) as archive_handle:
        archive_handle.extractall(go_root, filter='data')
go_bin = go_root / 'go' / 'bin'

elan_home = TOOLS / 'elan'
env = dict(os.environ, ELAN_HOME=str(elan_home))
env['PATH'] = f"{elan_home / 'bin'}:{go_bin}:{TOOLS / 'bin'}:{env['PATH']}"
run(['elan', 'toolchain', 'install', 'leanprover/lean4:v4.29.0-rc1'], env=env)
run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], env=env)
run(['python', '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], env=env)

landrun_source = TOOLS / 'landrun-source'
exact_checkout('https://github.com/Zouuup/landrun', '811cfff51ceaf3d9843708aa6d22e9b84ccac8b4', landrun_source)
(TOOLS / 'bin').mkdir(exist_ok=True)
landrun_binary = TOOLS / 'bin' / 'landrun'
run([str(go_bin / 'go'), 'build', '-trimpath', '-o', str(landrun_binary), './cmd/landrun'], cwd=landrun_source, env=env)

comparator = TOOLS / 'comparator'
exact_checkout('https://github.com/leanprover/comparator', 'ae061f79cdf7af458a26348177cfbd62da0123f6', comparator)
run(['lake', '-Kjobs=1', 'build', 'lean4export', 'comparator'], cwd=comparator, env=env)

exact_checkout(
    'https://github.com/leanprover-community/mathlib4',
    '1bc7728a050fc18ca2683f614c531cd7050ff063',
    CACHE_BASE,
)
assert CACHE_BASE.joinpath('lean-toolchain').read_text().strip() == 'leanprover/lean4:v4.29.0-rc1'
run(['lake', 'exe', 'cache', 'get'], cwd=CACHE_BASE, env=env)
run(['lake', 'build', '--no-build', 'Mathlib'], cwd=CACHE_BASE, env=env)
if not CACHE.exists():
    shutil.copytree(CACHE_BASE, CACHE, symlinks=True)

env['PATH'] = f"{TOOLS / 'bin'}:{env['PATH']}"
os.environ.update(env)
from vrm.config import load_protocol_amendment, load_study_config
from vrm.lean import cache_digest
study_config = load_study_config(PROJECT / 'configs/study.json')
amendment_path = PROJECT / 'protocol/runtime_provenance_amendment_2026-09-13.json'
amendment = load_protocol_amendment(study_config['protocol_amendment'], PROJECT / 'configs/study.json')
assert amendment_path.is_file()
CACHE_SHA256 = cache_digest(CACHE)
LANDRUN_SHA256 = hashlib.sha256(landrun_binary.read_bytes()).hexdigest()
provision_receipt = {
    'project_commit': PROJECT_GIT_REV,
    'protocol_amendment': study_config['protocol_amendment'],
    'cache_sha256': CACHE_SHA256,
    'landrun_sha256': LANDRUN_SHA256,
    'comparator_commit': 'ae061f79cdf7af458a26348177cfbd62da0123f6',
    'lean_version': 'v4.29.0-rc1',
    'mathlib_cache_source': 'official Azure cache via lake exe cache get',
    'go_version': go_version,
}
Path('/content/vrm-artifacts').mkdir(parents=True, exist_ok=True)
Path('/content/vrm-artifacts/provision-receipt.json').write_text(json.dumps(provision_receipt, sort_keys=True))

def export_artifacts(label):
    artifacts_root = Path('/content/vrm-artifacts')
    manifest_path = artifacts_root / 'artifact-manifest.json'
    artifact_hashes = {
        path.relative_to(artifacts_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(artifacts_root.rglob('*'))
        if path.is_file() and path != manifest_path
    }
    artifact_manifest = {
        'label': label,
        'project_commit': PROJECT_GIT_REV,
        'protocol_amendment': study_config['protocol_amendment'],
        'files': artifact_hashes,
    }
    manifest_path.write_text(json.dumps(artifact_manifest, sort_keys=True))
    archive_base = Path('/content') / f'vrm-{label}-{PROJECT_GIT_REV[:12]}'
    archive_path = Path(shutil.make_archive(str(archive_base), 'gztar', root_dir=artifacts_root))
    files.download(str(archive_path))
    return {'archive': str(archive_path), 'sha256': hashlib.sha256(archive_path.read_bytes()).hexdigest()}

provision_receipt


## 2. Run the trusted-verifier preflight

Colab normally starts as root, so security-sensitive commands run as a dedicated unprivileged account. This must return `ready`; otherwise stop before loading Gemma.

In [ ]:
subprocess.run(['id', 'vrmrunner'], capture_output=True).returncode == 0 or run(['useradd', '--create-home', 'vrmrunner'])
run(['mkdir', '-p', '/content/vrm-artifacts'])
run(['chown', '-R', 'vrmrunner:vrmrunner', '/content/vrm-artifacts'])

study_env = dict(os.environ)
study_env.update({
    'HOME': '/home/vrmrunner',
    'VRM_LEAN_EXECUTION_MODE': 'native',
    'VRM_LEAN_CACHE': str(CACHE),
    'VRM_LEAN_CACHE_SHA256': CACHE_SHA256,
    'VRM_LANDRUN_SHA256': LANDRUN_SHA256,
    'VRM_COMPARATOR_ROOT': str(comparator),
})
def as_study_user(command):
    return subprocess.run(
        ['runuser', '--preserve-environment', '-u', 'vrmrunner', '--', *command],
        env=study_env, text=True, capture_output=True,
    )

preflight = as_study_user([
    'vrm', 'preflight', '--execution-mode', 'native',
    '--cache-dir', str(CACHE), '--cache-sha256', CACHE_SHA256,
    '--landrun-sha256', LANDRUN_SHA256, '--comparator-root', str(comparator),
])
print(preflight.stdout)
assert preflight.returncode == 0, preflight.stderr


## 3. Restore and verify the frozen development population

The approved amendment preserves the already prepared corpus instead of constructing a new one. Upload `vrm-prepared-v2-approved.tar.gz` once when prompted. The package verifies the historical manifest and every public and private artifact against the amendment before any model output is generated.


In [ ]:
from vrm.data import validate_prepared_artifacts

if not PREPARED.exists():
    previous_directory = Path.cwd()
    os.chdir('/content')
    try:
        uploaded = files.upload()
    finally:
        os.chdir(previous_directory)
    assert PREPARED_ARCHIVE_NAME in uploaded, f'Upload {PREPARED_ARCHIVE_NAME}.'
    uploaded_archive = Path('/content') / PREPARED_ARCHIVE_NAME
    PREPARED.mkdir(parents=True)
    with tarfile.open(uploaded_archive) as archive_handle:
        archive_handle.extractall(PREPARED, filter='data')
manifest = validate_prepared_artifacts(PREPARED, amendment)
assert manifest['status'] == 'ready' and manifest['counts']['dev'] == 32
manifest['counts']


## 4. Run the registered real verifier acceptance suite

Before Gemma is loaded, this checks one known-valid proof, four distinct rejection paths, one filesystem escape attempt, and cache-mutation detection. The final mutation uses the disposable Mathlib cache and restores it from the separately preserved official-cache copy before the smoke.


In [ ]:
public_rows = [json.loads(line) for line in PREPARED.joinpath('dev.jsonl').read_text().splitlines()]
private_rows = [json.loads(line) for line in PREPARED.joinpath('verifier_metadata.jsonl').read_text().splitlines()]
private_by_id = {row['task_id']: row['metadata'] for row in private_rows if row['split'] == 'dev'}
target_public = next(row for row in public_rows if row['full_name'] == 'Nat.abundant_twelve')
target_private = private_by_id[target_public['task_id']]
task = {key: target_public[key] for key in ('task_id', 'repo_url', 'repo_commit', 'file_path', 'full_name')}
task['verifier'] = {
    **{key: target_private[key] for key in (
        'source_sha256', 'trace_sha256', 'start', 'end', 'proof_start',
        'proof_end', 'theorem_statement',
    )},
    'benchmark_doi': '10.5281/zenodo.18815372',
    'cache_sha256': CACHE_SHA256,
}
reference_proof = target_private['reference_proof']
assert reference_proof == 'by\n  decide'
escape_path = Path('/tmp/vrm-sandbox-escape')
assert not escape_path.exists(), 'Use a fresh runtime for the sandbox test.'
mutation_target = next(
    path for path in CACHE.rglob('*.olean')
    if '.lake' in path.parts and path.is_file() and not path.is_symlink()
)
real_cases = {
    'valid': {'task': task, 'proof': reference_proof, 'expected': 'valid'},
    'invalid': {'task': task, 'proof': 'by\n  contradiction', 'expected': 'invalid'},
    'self_reference': {'task': task, 'proof': 'by\n  exact Nat.abundant_twelve', 'expected': 'invalid'},
    'sorry': {'task': task, 'proof': 'by\n  sorry', 'expected': 'invalid'},
    'unauthorized_axiom': {'task': task, 'proof': 'by\n  native_decide', 'expected': 'invalid'},
    'sandbox_escape': {
        'task': task,
        'proof': 'by\n  run_tac\n    IO.FS.writeFile \"/tmp/vrm-sandbox-escape\" \"escaped\"\n  decide',
        'expected': 'invalid',
    },
    'cache_mutation': {
        'disposable_cache_dir': str(CACHE),
        'relative_path': str(mutation_target.relative_to(CACHE)),
        'original_cache_sha256': CACHE_SHA256,
        'allow_mutation': True,
    },
}
cases_path = Path('/content/vrm-artifacts/real-verifier-cases.json')
cases_path.write_text(json.dumps(real_cases, sort_keys=True))
run(['chown', '-R', 'vrmrunner:vrmrunner', str(CACHE), str(cases_path)])
real_results = Path('/content/vrm-artifacts/verifier-results')
real_results.mkdir(exist_ok=True)
run(['chown', '-R', 'vrmrunner:vrmrunner', str(real_results)])
real_env = dict(
    study_env,
    VRM_RUN_REAL_LEAN_TESTS='1',
    VRM_LEAN_REAL_CASES=str(cases_path),
    VRM_LEAN_REAL_RESULTS=str(real_results),
)
junit_path = Path('/content/vrm-artifacts/real-verifier-tests.xml')
real_tests = None
try:
    real_tests = subprocess.run(
        [
            'runuser', '--preserve-environment', '-u', 'vrmrunner', '--',
            'python', '-m', 'pytest',
            'tests/test_lean.py::TestRealComparatorOptIn', '-q',
            f'--junitxml={junit_path}',
        ],
        env=real_env, text=True, capture_output=True,
    )
finally:
    shutil.rmtree(CACHE)
    shutil.copytree(CACHE_BASE, CACHE, symlinks=True)
    run(['chmod', '-R', 'a-w', str(CACHE)])
    assert cache_digest(CACHE) == CACHE_SHA256
assert real_tests is not None
print(real_tests.stdout)
if real_tests.stderr:
    print(real_tests.stderr)
sandbox_passed = not escape_path.exists()
results_complete = len(list(real_results.glob('*.json'))) == 7
verifier_passed = sandbox_passed and results_complete and real_tests.returncode == 0
verifier_export = export_artifacts(
    'verifier-acceptance' if verifier_passed else 'verifier-failure'
)
assert verifier_passed, {
    'sandbox_passed': sandbox_passed,
    'results_complete': results_complete,
    'returncode': real_tests.returncode,
}


## 5. Run or resume the registered 32 x 4 smoke

One process loads Gemma once, first generates and verifies exactly one complete candidate, and continues only if that integration probe succeeds. It then resumes the same immutable run through all 128 candidates. The probe is recorded separately and never evaluates the scientific feasibility gate.

In [ ]:
smoke = as_study_user([
    'vrm', 'smoke', '--config', str(PROJECT / 'configs/study.json'),
    '--prepared-dir', str(PREPARED), '--output', str(RUN),
    '--execution-mode', 'native', '--cache-dir', str(CACHE),
    '--cache-sha256', CACHE_SHA256, '--landrun-sha256', LANDRUN_SHA256,
    '--comparator-root', str(comparator),
])
print(smoke.stdout)
if smoke.stderr:
    print(smoke.stderr)
post_smoke_cache_sha256 = cache_digest(CACHE)
assert post_smoke_cache_sha256 == CACHE_SHA256, 'Verifier cache changed during the smoke.'
assert RUN.joinpath('summary.json').is_file(), 'No scientific summary was produced.'
summary = json.loads(RUN.joinpath('summary.json').read_text())
summary


## 6. Apply the mechanical gate

Continue to protected monitor work only when all 128 candidates completed and the frozen valid, invalid, and mixed-task thresholds passed. Operational failures leave H1-H3 untested.

In [ ]:
decision = {
    'run_status': summary['status'],
    'completed_candidates': summary['completed_candidates'],
    'failed_attempts': summary['failed_attempts'],
    'feasibility': summary['feasibility'],
    'continue_to_H1_H3': bool(
        summary['status'] == 'completed'
        and summary['completed_candidates'] == 128
        and summary['feasibility']
        and summary['feasibility']['passed']
    ),
}
Path('/content/vrm-artifacts/development-decision.json').write_text(
    json.dumps(decision, sort_keys=True)
)
development_export = export_artifacts('development-smoke')
decision
